# WBSNet — Option-A Paper Run on Colab

End-to-end runner for the **Option-A paper scope**:

- 7 ablation variants (A1-A7) × seeds on **Kvasir-SEG**
- Full WBSNet on **CVC-ClinicDB** and **ISIC2018**
- U-Net baselines on Kvasir, ClinicDB, ISIC2018
- Generalization eval: Kvasir checkpoint → CVC-ColonDB
- Aggregation, significance tests, complexity, qualitative figures

**Recommended runtime:** Colab Pro+ A100 with background execution.
**Backup:** Colab Pro L4 for smoke tests or smaller runs.

Before running:
1. `Runtime → Change runtime type → A100 GPU` (or L4 if on Pro).
2. `Runtime → Background execution` (Pro+ only) — lets you close the laptop.
3. Have the processed dataset on Google Drive at `MyDrive/WBSNet_Dataset/` with subdirs `kvasir/`, `cvc_clinicdb/`, `cvc_colondb/`, `isic2018/`.
4. If continuing from the Kaggle seed-3407 run, keep the exported `wbsnet_paper_runs/` folder at `MyDrive/wbsnet_paper_runs/`.
5. The expected processed layout is split-based: `kvasir/train/images`, `kvasir/val/masks`, `isic2018/test/images`, `cvc_colondb/test/masks`, etc.

Each section is independent — re-run only what you need.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Environment check

In [ ]:
!nvidia-smi

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Switch runtime to A100 (Pro+) or L4 (Pro).")

gpu_name = torch.cuda.get_device_name(0)
print(f"\nDetected GPU: {gpu_name}")
SUPPORTED = ("A100", "L4", "H100")
if not any(tag in gpu_name for tag in SUPPORTED):
    raise RuntimeError(
        f"Unsupported GPU '{gpu_name}'. The paper pipeline targets A100/L4/H100. "
        "Change via Runtime -> Change runtime type before running cell 19."
    )

## 2. Clone repo

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/MrArrogant2002/WBSNet-research-paper.git"
REPO_DIR = Path("/content/WBSNet-research-paper")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}")

os.chdir(REPO_DIR)
!git fetch origin main
!git reset --hard origin/main

print("CWD:", os.getcwd())
print("Runtime repo commit:")
!git rev-parse --short HEAD


## 3. Install dependencies

Colab already ships with CUDA-enabled PyTorch. Install the Colab-compatible dependency set, then install this repo with `--no-deps` so editable install does not downgrade the runtime image.

In [ ]:
# Keep Colab's CUDA-enabled PyTorch. The second install uses --no-deps so pip does not downgrade torch.
!python3 -m pip install --quiet -r requirements-colab.txt
!python3 -m pip install --quiet --no-deps -e .

import torch
print("torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


## 4. Verify repo structure

In [ ]:
!python3 scripts/verify_repo.py

## 5. Mount Google Drive and link dataset

If your dataset is laid out differently, edit `DRIVE_DATASET` and `DATASET_MAP` below.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_DATASET = Path("/content/drive/MyDrive/WBSNet_Dataset")
assert DRIVE_DATASET.exists(), f"Dataset not found at {DRIVE_DATASET}. Edit DRIVE_DATASET in this cell."

DATA_ROOT = REPO_DIR / "data"
DATA_ROOT.mkdir(exist_ok=True)

DATASET_MAP = {
    "kvasir": "Kvasir-SEG",
    "cvc_clinicdb": "CVC-ClinicDB",
    "cvc_colondb": "CVC-ColonDB",
    "isic2018": "ISIC2018",
}

for src_name, dst_name in DATASET_MAP.items():
    src = DRIVE_DATASET / src_name
    dst = DATA_ROOT / dst_name
    if not src.exists():
        print(f"MISSING: {src}")
        continue
    if dst.is_symlink():
        dst.unlink()
    elif dst.exists():
        print(f"EXISTS: {dst} is not a symlink; leaving it unchanged")
        continue
    os.symlink(src, dst)
    print(f"LINKED: {src} -> {dst}")

print("\nDataset directory links:")
!find data -maxdepth 2 -type l -ls || true


## 6. Restore previous outputs and import legacy / Kaggle-session artifacts

Run this **before** the paper pipeline. It pulls every prior result that lives on Drive into `outputs/` so already-completed runs are skipped instead of retrained:

| Drive root | What it holds |
|---|---|
| `wbsnet_paper_runs/` | Original Kaggle seed-3407 — Kvasir A1–A7, CVC-ClinicDB A1–A7, ISIC2018 A1 |
| `wbsnet_kaggle_session1/` | Kaggle Session 1 — ISIC2018 A2 seed 3407 |
| `wbsnet_kaggle_session2/` | Kaggle Session 2 — Kvasir A2/A5/A6/A7 seed 3408 |
| `wbsnet_kaggle_session3/` | Kaggle Session 3 — Kvasir A1/A3/A4 + ClinicDB A1/A2 seed 3408 |
| `WBSNet_outputs/` | Any partial Colab run from a previous session |

See `kaggle-session-plan.md` in the repo root for the strategy behind the three Kaggle sessions and what work remains for this Colab notebook to handle.

In [ ]:
from pathlib import Path

DRIVE_OUTPUTS = Path("/content/drive/MyDrive/WBSNet_outputs")

# All Drive roots that may hold importable runs in the legacy
# `paper_suite/<dataset>/<variant>/seed_<seed>/<run_name>/` layout.
# Order matters: later entries can overwrite earlier ones if they contain
# the same run, so list newer / higher-quality sources later.
LEGACY_ROOTS = [
    Path("/content/drive/MyDrive/wbsnet_paper_runs"),       # original Kaggle seed-3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session1"),  # ISIC2018 A2 seed 3407
    Path("/content/drive/MyDrive/wbsnet_kaggle_session2"),  # Kvasir seed 3408 subset
    Path("/content/drive/MyDrive/wbsnet_kaggle_session3"),  # Kvasir/ClinicDB seed 3408 subset
]

if DRIVE_OUTPUTS.exists():
    print(f"Restoring previous Colab outputs from {DRIVE_OUTPUTS}")
    !mkdir -p outputs
    !rsync -a --info=progress2 {DRIVE_OUTPUTS}/ outputs/
else:
    print(f"No previous Colab output backup found at {DRIVE_OUTPUTS}")

imported_any = False
for legacy_root in LEGACY_ROOTS:
    if not legacy_root.exists():
        print(f"  skip: {legacy_root} not present on Drive")
        continue
    print(f"\nImporting from {legacy_root}")
    !python3 scripts/import_legacy_paper_runs.py \
        --legacy-root {legacy_root} \
        --output-root outputs \
        --verify-forward
    imported_any = True

if not imported_any:
    print("\nNo legacy / Kaggle-session folders found on Drive; starting from scratch.")

print("\nExisting best checkpoints that will be skipped by the paper runner:")
!test -d outputs && find outputs -path '*/checkpoints/best.pt' | sort | sed -n '1,120p' || true


## 7. (Optional) W&B login

Skip this cell to run with W&B offline. Set `WANDB_API_KEY` to log online.

In [ ]:
import os

# Preferred: store the key once via the key icon in the left sidebar (Colab Secrets)
# as 'WANDB_API_KEY', then load it here. Falls back gracefully if the secret is absent.
try:
    from google.colab import userdata  # type: ignore[import-not-found]

    key = userdata.get("WANDB_API_KEY")
    if key:
        os.environ["WANDB_API_KEY"] = key
        print("WANDB_API_KEY loaded from Colab Secrets.")
    else:
        print("Colab Secret 'WANDB_API_KEY' not set; W&B will run offline unless you set it below.")
except Exception as exc:  # noqa: BLE001
    print(f"Colab userdata unavailable ({exc}); W&B will run offline unless you set it below.")

# Alternative paths (uncomment one if you do not want to use Colab Secrets):
#  - Hardcode for a single session (NOT recommended; never commit this):
#       os.environ["WANDB_API_KEY"] = "YOUR_WANDB_KEY_HERE"
#  - Place a `.env` file at the repo root with WANDB_API_KEY=...; ExperimentLogger auto-loads it.
#  - Interactive login that does not echo the key:
#       from getpass import getpass; os.environ["WANDB_API_KEY"] = getpass("WANDB key: ")
# Then, to actually log in to the W&B CLI for this session:
# !wandb login --relogin


## 8. Smoke test (1 epoch, batch 2)

Confirms the pipeline runs end-to-end. ~2 min on A100, ~3 min on L4.

In [ ]:
# Smoke test: 1 epoch, tiny batch. Run this before the full paper run.
!python3 train.py --config configs/kvasir_wbsnet.yaml \
    --override train.epochs=1 train.batch_size=2 train.grad_accum_steps=1 \
                dataset.split_strategy=pre_split_dirs \
                dataset.num_workers=2 dataset.prefetch_factor=2 \
                evaluation.compute_hd95=false evaluation.max_visualizations=0 \
                runtime.device=cuda runtime.wandb.mode=offline \
                experiment.name=smoke_test \
                experiment.run_name=smoke_test


## 9. Run the full Option-A paper pipeline

**This is the headline cell.** It runs every config across the seeds you specify, then the ColonDB generalization eval, then aggregation/significance/complexity. Already-completed runs (whose `best.pt` exists, including those imported from the Kaggle sessions in Section 6) are skipped — safe to re-run.

**Continuation plan after the three Kaggle sessions land** (per `kaggle-session-plan.md`):

| Status after Sessions 1–3 | What still needs Colab |
|---|---|
| Seed 3407: 100% complete (Kvasir A1–A7, ClinicDB A1–A7, ISIC2018 A1+A2) | nothing |
| Seed 3408: Kvasir A1–A7 done, ClinicDB A1+A2 done, **ISIC2018 A1+A2 missing** | ~10–14 h A100 (~130–180 units) |
| Seed 3409: nothing done | ~22–31 h A100 (~290–400 units) — drop if budget is tight |

**Recommended:** run with `--seeds 3407 3408` to ship an n=2 paper. Add seed 3409 only if you have ≥400 spare units.

**Pro+ tip:** enable Background Execution so the cell keeps running with the laptop closed.

**Stability preset:** this cell uses batch 8 with gradient accumulation 2, lower learning rates, and bounded non-finite-gradient skip recovery. Existing `best.pt` checkpoints are still skipped; failed partial runs restart cleanly.

In [ ]:
# Continuation paper run. Existing best.pt checkpoints are skipped automatically.
# A background rsync mirrors outputs/ to Drive every 10 minutes so progress
# survives a Colab disconnect mid-run. Final flush runs in the `finally` block.
#
# Budget preset: seeds 3407 and 3408. Add 3409 only if you are ready to spend
# substantially more compute units.

import os
import signal
import subprocess
from pathlib import Path

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)

sync_cmd = (
    "while true; do "
    f"  rsync -a outputs/ {DRIVE_BACKUP}/ >> /content/rsync.log 2>&1; "
    "  sleep 600; "
    "done"
)
sync_proc = subprocess.Popen(
    ["bash", "-c", sync_cmd],
    start_new_session=True,
)
print(f"Started periodic Drive sync (PID {sync_proc.pid}); logs at /content/rsync.log")

try:
    !python3 scripts/run_paper_optionA.py \
        --seeds 3407 3408 \
        --no-auto-resume \
        --override train.epochs=150 train.batch_size=8 train.grad_accum_steps=2 \
                   train.encoder_lr=0.00005 train.decoder_lr=0.0005 \
                   train.nonfinite_grad_action=skip train.max_nonfinite_grad_steps=10 \
                   dataset.split_strategy=pre_split_dirs \
                   dataset.num_workers=2 dataset.prefetch_factor=2 \
                   train.save_every=5 runtime.device=cuda runtime.wandb.mode=offline \
                   evaluation.max_visualizations=8
finally:
    try:
        os.killpg(os.getpgid(sync_proc.pid), signal.SIGTERM)
        print("Stopped periodic Drive sync.")
    except ProcessLookupError:
        pass
    print("Final rsync to Drive...")
    !rsync -a --info=progress2 outputs/ {DRIVE_BACKUP}/


## 10. Generate qualitative paper figures

After training, build the qualitative comparison panels for the paper.

In [ ]:
from pathlib import Path

candidates = sorted(
    Path("outputs").rglob("checkpoints/best.pt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
kvasir_runs = [p for p in candidates if "kvasir_wbsnet" in str(p) and "baseline" not in str(p)]
if not kvasir_runs:
    raise RuntimeError("No Kvasir WBSNet checkpoint found.")

BEST_CKPT = kvasir_runs[0]
print("Using checkpoint:", BEST_CKPT)
PRED_DIR = BEST_CKPT.parent.parent / "predictions"

# The processed Kvasir card may not include a test split, so use val for qualitative panels.
if not PRED_DIR.exists() or not any(PRED_DIR.iterdir()):
    !python3 predict.py --config configs/kvasir_wbsnet.yaml \
        --checkpoint {BEST_CKPT} --split val \
        --override dataset.split_strategy=pre_split_dirs \
                   dataset.num_workers=2 runtime.wandb.mode=offline \
                   evaluation.max_visualizations=8

!python3 scripts/make_paper_figures.py \
    --input-dir {PRED_DIR} --limit 8 --columns 2


## 11. Save outputs back to Drive

Colab disks reset on disconnect. Always rsync after any major run.

In [ ]:
from pathlib import Path

DRIVE_BACKUP = Path("/content/drive/MyDrive/WBSNet_outputs")
DRIVE_BACKUP.mkdir(parents=True, exist_ok=True)
!rsync -a --info=progress2 outputs/ {DRIVE_BACKUP}/
print(f"Backed up outputs to {DRIVE_BACKUP}")


## 12. Build the paper PDF (optional, can also be done locally)

In [ ]:
# Uncomment to install LaTeX and build the PDF inside Colab.
# !apt-get install -y --quiet texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-publishers
# !cd paper && pdflatex paper && bibtex paper && pdflatex paper && pdflatex paper